# Module 2.1: Compare GraphRAG Read Paths

**Purpose:** Compare ways to read the graph and choose the right path for each question.

**Overview**

- **Semantic search:** Finds source text with a meaning similar to the question.
- **Exact-term search:** Finds source text with an exact word, name, or number.
- **Graph expansion:** Follows relationships from a matching source to related facts.
- **Structured query:** Filters, counts, or calculates values across graph records.
- **Provenance:** Shows which source document supports a result.

The examples start with simple text search. They then add graph facts and structured queries. Each example checks the retrieved records and their sources.

The final example compares passage search with Text2Cypher. Module 3 gives both paths to one agent.

## Check that Module 1 finished

Finish Module 1 before you start this notebook. Then run the cells in order.

- **Read-only notebook:** The examples read the graph and leave your data unchanged.
- **Failed check:** Return to Module 1 and run every cell. Then restart this notebook from the top.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("02-connected-context")
print(f'Workshop root: {REPO_ROOT}')

In [ ]:
from IPython.display import HTML, display
from neo4j import GraphDatabase, READ_ACCESS
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from workshop import hybrid_retrieval
from workshop.agent_tools import (
    EXPECTED_QUERY_ERRORS,
    query_hotel_records,
    search_hotel_passages,
)
from workshop.aws_region import aws_region, configure_aws_region
from workshop.bedrock_providers import BedrockEmbeddings
from workshop.contracts import HYBRID_TOP_K
from workshop.graph_connection import (
    graph_database,
    neo4j_auth,
    neo4j_uri,
    require_neo4j_env,
)
from workshop.graph_schema import GRAPH_SCHEMA
from workshop.hybrid_retrieval import (
    MAX_GRAPH_QUERY_RECORDS,
    graph_query,
    search_hotel_knowledge,
)
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    EMBEDDING_DIMENSIONS,
)
from workshop.retrieval_setup import (
    CHICAGO_CITY,
    CHICAGO_EXCLUSION,
    CHICAGO_FILTER_QUERY,
    CHICAGO_QUALIFIER,
    CHICAGO_SOURCE_FILES,
    chicago_filter_problems,
    chicago_filter_records,
    fixture_for,
    graph_context_problems,
    report_problems,
    source_context_problems,
    source_fixture_problems,
    verify_retrieval_indexes,
)

configure_aws_region()
require_neo4j_env()
DATABASE = graph_database()
# neo4j-graphrag still calls the deprecated db.index.vector.queryNodes procedure.
# Hide that deprecation notice. Keep every other warning visible.
driver = GraphDatabase.driver(
    neo4j_uri(),
    auth=neo4j_auth(),
    notifications_disabled_classifications=["DEPRECATION"],
)
driver.verify_connectivity()
print(f'Connected to Neo4j database: {DATABASE}')

## Check that the graph has the required data

Run the next cell before the retrieval examples. It checks the source documents, embeddings, graph facts, relationships, and indexes used in this notebook.

A failed check explains how to return to Module 1 and prepare the graph.

In [ ]:
verify_retrieval_indexes(driver)
print(f'PASS  {CHUNK_VECTOR_INDEX}: online, cosine, {EMBEDDING_DIMENSIONS} dimensions')
print(f'PASS  {CHUNK_FULLTEXT_INDEX}: online over Chunk.text')

problems = source_fixture_problems(driver)
problems.extend(chicago_filter_problems(chicago_filter_records(driver)))
if problems:
    problems.append(
        'Return to the Module 1 notebook, run every cell, then rerun this '
        'notebook from the top. This check did not modify the graph.'
    )
report_problems(
    problems,
    'Cairo and Chicago source, path, field, amenity, and filter fixtures',
)

## Inspect the graph structure and search settings

Run the next cell to see the relationships that connect hotel facts. The retrieval examples use these paths to add facts and provenance to search results.

- **`GRAPH_SCHEMA`:** Lists the node labels and relationships used across the workshop.
- **Embedding:** Uses the same Amazon Nova model and 1024 dimensions that created the stored vectors.
- **Database:** Sends every query to the configured Neo4j database.

In [ ]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Text provenance: <code>(:Chunk)-[:FROM_DOCUMENT]-&gt;(:Document)</code>.</p>'
    '<p>Entity provenance: <code>(:Hotel)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

## Set up one display for all results

Run the next cell to define a shared result format. The shared format makes the retrieval examples easier to compare.

- **Question:** Shows the input given to the retriever.
- **Retriever:** Names the search method.
- **Ranking:** Shows the top-k limit, rank, and score.
- **Source:** Shows the source filename and complete `Chunk.text`.
- **Context size:** Counts structured text and source text separately.
- **Missing fields:** Lists the requested fields that a result does not contain.

The graph provenance path supplies each source filename.

In [ ]:
def source_for_chunk_id(chunk_id):
    with driver.session(database=DATABASE, default_access_mode=READ_ACCESS) as session:
        records = session.run(
            '''
            CYPHER 25
            MATCH (matched:Chunk)-[:FROM_DOCUMENT]->(document:Document)
            WHERE elementId(matched) = $chunk_id
            RETURN DISTINCT document.source_filename AS source_filename
            ORDER BY source_filename
            ''',
            chunk_id=chunk_id,
        )
        return ', '.join(record['source_filename'] for record in records)


def context_value_chars(value):
    if value is None:
        return 0
    if isinstance(value, dict):
        return sum(
            context_value_chars(key) + context_value_chars(item)
            for key, item in value.items()
        )
    if isinstance(value, (list, tuple, set)):
        return sum(context_value_chars(item) for item in value)
    return len(str(value))


def context_char_counts(structured_fields, source_text):
    structured_chars = sum(
        context_value_chars(value) for value in structured_fields.values()
    )
    return structured_chars, len(source_text)


def print_record(record, fields):
    for field in fields:
        print(f'  {field}: {record.get(field)}')


def text_result_formatter(record):
    node = record.get('node') or {}
    chunk = node.get('text') or ''
    chunk_id = record.get('elementId')
    return RetrieverResultItem(
        content=chunk,
        metadata={
            'score': record.get('score'),
            'source_filename': source_for_chunk_id(chunk_id) if chunk_id else '',
        },
    )


embedder = BedrockEmbeddings(region_name=aws_region())


def context_rows(
    question,
    retriever_name,
    result,
    top_k,
    configuration,
    required_terms,
    absent_fields=(),
):
    """Print every retrieval result the same way and return the rows.

    `required_terms` maps each field to text that must appear in the source
    chunk. `absent_fields` lists fields that only graph expansion can add.
    """
    print(f'Question: {question}')
    print(f'Retriever: {retriever_name}')
    print(f'Configuration: {configuration}')
    print(f'Top-k: {top_k}')
    print(f'Result count: {len(result.items)}')
    rows = []
    for rank, item in enumerate(result.items, 1):
        metadata = item.metadata or {}
        chunk = str(item.content or '')
        source_text = chunk.casefold()
        missing = [] if metadata.get('source_filename') else ['source_filename']
        missing.extend(
            field for field, term in required_terms.items()
            if term.casefold() not in source_text
        )
        missing.extend(absent_fields)
        structured_chars, source_text_chars = context_char_counts(
            {'source_filename': metadata.get('source_filename')}, chunk
        )
        row = {
            'rank': rank,
            'score': metadata.get('score'),
            'source_filename': metadata.get('source_filename'),
            'structured_context_chars': structured_chars,
            'source_text_chars': source_text_chars,
            'missing_requested_fields': missing,
            'chunk': chunk,
        }
        rows.append(row)
        score = row['score']
        score_text = 'n/a' if score is None else f'{score:.6f}'
        print(f'Rank {rank} | score={score_text} | source={row["source_filename"]}')
        print(f'Structured-field context: {structured_chars} characters')
        print(f'Source-text context: {source_text_chars} characters')
        print(f'Missing requested fields: {missing or "none"}')
        print('Complete Chunk text:')
        print(chunk)
        print()
    return rows

## 1. Find an arrival time with semantic search

Run the next cell to find a source by meaning. The question uses "arrival processing" while the source uses `Standard check-in time`. `VectorRetriever` can connect these phrases because they have a similar meaning.

- **Input:** Asks for the arrival time at AnyCompany Cairo Nile View.
- **Expected result:** Returns the Cairo source with the supported time `3:00 PM`.
- **Check:** Verifies the retrieved context and leaves answer generation out of scope.
- **Workshop chunk:** Returns the full hotel document because Module 1 stores each hotel in one large chunk. A production system can use smaller retrieval chunks that fit its sources and context limit.

In [ ]:
CAIRO_FIXTURE = fixture_for('hotel-cairo-001.txt')
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
ARRIVAL_QUESTION = (
    'When does standard arrival processing begin at AnyCompany Cairo Nile View?'
)
VECTOR_TOP_K = 3
arrival_result = vector_retriever.search(
    query_text=ARRIVAL_QUESTION,
    top_k=VECTOR_TOP_K,
)
arrival_rows = context_rows(
    ARRIVAL_QUESTION,
    'VectorRetriever',
    arrival_result,
    VECTOR_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, cosine, Nova {EMBEDDING_DIMENSIONS} dimensions',
    {'supported_arrival_time': '3:00 PM'},
)
report_problems(
    source_context_problems(arrival_rows, CAIRO_FIXTURE),
    'Cairo source and supported 3:00 PM arrival time are visible.',
)

## 2. Find a postal code with hybrid search

Run the next cell to compare semantic search with hybrid search. Both searches return at most five results.

The postal code `60611` is an exact identifier. Full-text search matches these exact characters. Hybrid search combines that match with semantic similarity, so it can rank the Chicago source more reliably.

- **Vector signal:** The complete question supplies the semantic search input.
- **Full-text signal:** `60611` supplies the exact-term search input.
- **Hybrid ranking:** The reviewed linear ranker combines both signals with `alpha=0.2`.
- **Shared vector:** Both searches use the same question vector.
- **Output:** Shows the rank, source filename, and presence of `60611` for each result. The full-text term causes the ranking difference.
- **Expected result:** The hybrid results include the Chicago hotel, postal code, and cancellation policy.

In [ ]:
CHICAGO_IDENTIFIER_FIXTURE = fixture_for('hotel-chicago-001.txt')
CANCELLATION_TERM = 'at least 24 hours prior to arrival'
IDENTIFIER_QUESTION = 'What is the cancellation policy for the hotel at 60611?'
IDENTIFIER_TOP_K = 5

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)

vector_identifier_result = vector_retriever.search(
    query_text=IDENTIFIER_QUESTION,
    top_k=IDENTIFIER_TOP_K,
)
# Use the same question vector in both searches. This isolates the effect
# of the full-text term added by hybrid search.
hybrid_identifier_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=IDENTIFIER_TOP_K,
    ranker='linear',
    alpha=0.2,
)


def identifier_rows(result):
    """Return the source filename and text of each result, in rank order.

    This comparison needs only the source filename and text. Reading those
    fields directly keeps the ranking output short.
    """
    return [
        {
            'rank': rank,
            'source_filename': (item.metadata or {}).get('source_filename'),
            'chunk': str(item.content or ''),
        }
        for rank, item in enumerate(result.items, 1)
    ]


hybrid_identifier_rows = identifier_rows(hybrid_identifier_result)
print(f'Question: {IDENTIFIER_QUESTION}')
print(f'Top-k: {IDENTIFIER_TOP_K}')
print('Vector signal: complete question. Full-text term: 60611.')
print('Hybrid ranking: linear ranker, alpha=0.2.')
for label, rows in (
    ('Vector', identifier_rows(vector_identifier_result)),
    ('Hybrid', hybrid_identifier_rows),
):
    for row in rows:
        print(
            f'{label} rank {row["rank"]} | source={row["source_filename"]} | '
            f'contains 60611: {"60611" in row["chunk"]}'
        )
report_problems(
    source_context_problems(
        hybrid_identifier_rows,
        CHICAGO_IDENTIFIER_FIXTURE,
        extra_terms=(CANCELLATION_TERM,),
    ),
    'Hybrid context contains the hotel, postal code, and cancellation policy.',
)

## 3. Find a source, then add related hotel facts

Run the next cell to combine semantic search with a fixed graph query. `VectorCypherRetriever` first finds a matching `Chunk`. It then follows relationships to the hotel and its amenities.

- **Source record:** Keeps the matching `Chunk`, its `Document`, and the semantic score.
- **Graph facts:** Adds the hotel name, hotel ID, guest rating, and amenities.
- **Provenance:** Shows the graph path used to return each field.
- **Comparison:** Runs plain vector search first, then shows the structured fields added by graph expansion.
- **Extraction limit:** Returns only facts that extraction wrote to Neo4j. Compare the graph fields with the source text to find missing or merged facts.

In [ ]:
VECTOR_CYPHER_QUERY = '''
MATCH (node)-[:FROM_DOCUMENT]->(document:Document)
OPTIONAL MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity)
WITH node, score, document, hotel, collect(DISTINCT amenity.name) AS amenities
RETURN hotel.name AS hotel_name,
       hotel.hotel_id AS hotel_id,
       hotel.guest_rating AS guest_rating,
       document.source_filename AS source_filename,
       amenities,
       node.text AS source_chunk,
       score AS semantic_score,
       CASE
           WHEN hotel IS NULL THEN ['FROM_DOCUMENT']
           ELSE ['FROM_DOCUMENT', 'FROM_CHUNK', 'OFFERS_AMENITY']
       END AS relationship_types,
       CASE
           WHEN hotel IS NULL THEN 'missing Hotel enrichment for semantic hit'
           ELSE 'complete Hotel enrichment'
       END AS graph_enrichment_status,
       {
           source_chunk: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           source_filename: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           hotel_name: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           hotel_id: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           guest_rating: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           amenities: '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)'
       } AS field_provenance
ORDER BY semantic_score DESC,
         CASE WHEN hotel_id IS NULL THEN 1 ELSE 0 END ASC,
         hotel_id ASC
'''

GRAPH_REQUESTED_FIELDS = (
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'source_filename',
    'amenities',
)
# Count these structured fields when comparing graph expansion with plain
# vector search.
GRAPH_STRUCTURED_FIELDS = (
    *GRAPH_REQUESTED_FIELDS,
    'relationship_types',
    'graph_enrichment_status',
    'field_provenance',
)
GRAPH_RECORD_FIELDS = (
    *GRAPH_REQUESTED_FIELDS,
    'semantic_score',
    'relationship_types',
    'graph_enrichment_status',
    'field_provenance',
    'missing_requested_fields',
    'structured_context_chars',
    'source_text_chars',
)
# Plain vector search returns only the source filename as a named field.
# A reader must extract the remaining facts from the source text.
VECTOR_NAMED_FIELDS = ('source_filename',)


def graph_result_formatter(record):
    metadata = {
        field: record.get(field) for field in GRAPH_REQUESTED_FIELDS
    }
    metadata['amenities'] = metadata['amenities'] or []
    metadata.update(
        semantic_score=record.get('semantic_score'),
        relationship_types=record.get('relationship_types') or [],
        graph_enrichment_status=record.get('graph_enrichment_status'),
        field_provenance=record.get('field_provenance') or {},
    )
    metadata['missing_requested_fields'] = [
        field for field in GRAPH_REQUESTED_FIELDS
        if metadata.get(field) is None or metadata.get(field) == []
    ]
    return RetrieverResultItem(
        content=record.get('source_chunk') or '',
        metadata=metadata,
    )


vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=VECTOR_CYPHER_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
    neo4j_database=DATABASE,
)
CAIRO_GRAPH_QUESTION = (
    'What amenities and guest rating does AnyCompany Cairo Nile View have?'
)
GRAPH_TOP_K = 3
graph_vector_result = vector_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_vector_rows = context_rows(
    CAIRO_GRAPH_QUESTION,
    'VectorRetriever',
    graph_vector_result,
    GRAPH_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic entry before graph expansion',
    {
        'hotel_name': CAIRO_FIXTURE.hotel_name,
        'guest_rating': f'{CAIRO_FIXTURE.guest_rating}/5.0',
        'amenities': 'Hotel Amenities',
    },
    absent_fields=('hotel_id',),
)
graph_result = vector_cypher_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)

graph_records = []
print(f'Question: {CAIRO_GRAPH_QUESTION}')
print('Retriever: VectorCypherRetriever')
print(f'Configuration: index={CHUNK_VECTOR_INDEX}, top_k={GRAPH_TOP_K}, reviewed traversal')
print(f'Result count: {len(graph_result.items)}')
for rank, item in enumerate(graph_result.items, 1):
    record = dict(item.metadata or {})
    record['source_chunk'] = str(item.content or '')
    record['rank'] = rank
    structured_chars, source_text_chars = context_char_counts(
        {field: record[field] for field in GRAPH_STRUCTURED_FIELDS},
        record['source_chunk'],
    )
    record['structured_context_chars'] = structured_chars
    record['source_text_chars'] = source_text_chars
    graph_records.append(record)
    print(f'Record {rank}:')
    print_record(record, GRAPH_RECORD_FIELDS)
    print('  source_chunk:')
    print(record['source_chunk'])

missing_enrichment = [
    record for record in graph_records
    if record['graph_enrichment_status'].startswith('missing')
]
print(
    f'Graph enrichment: {len(graph_records) - len(missing_enrichment)} complete, '
    f'{len(missing_enrichment)} missing Hotel context. Semantic hits without an '
    'extracted Hotel stay visible instead of disappearing.'
)

cairo_graph = next(
    record for record in graph_records
    if record['source_filename'] == CAIRO_FIXTURE.source_filename
)
vector_cairo = next(
    row for row in graph_vector_rows
    if row['source_filename'] == CAIRO_FIXTURE.source_filename
)
requested = len(GRAPH_REQUESTED_FIELDS)
graph_named = requested - len(cairo_graph['missing_requested_fields'])
print('Context comparison:')
print(
    f'  Vector: {len(VECTOR_NAMED_FIELDS)}/{requested} named fields, '
    f'{vector_cairo["structured_context_chars"]} structured and '
    f'{vector_cairo["source_text_chars"]} source-text characters'
)
print(
    f'  Vector-Cypher: {graph_named}/{requested} named fields, '
    f'{cairo_graph["structured_context_chars"]} structured and '
    f'{cairo_graph["source_text_chars"]} source-text characters'
)
print('Extraction quality limits graph enrichment. Missing extracted relationships stay missing.')
report_problems(
    graph_context_problems(graph_records, CAIRO_FIXTURE),
    'Cairo Vector-Cypher context includes every locked field and provenance path.',
)

## 4. Filter hotels with fixed Cypher

Run the next cell to find Chicago hotels that have both a spa and a swimming pool. A fixed Cypher query checks both conditions on each hotel.

- **Candidate:** Any hotel with an address that contains Chicago.
- **Qualifier:** A candidate with both required amenities.
- **Exclusion:** A candidate missing one or both required amenities.
- **Expected result:** The output shows two candidates, one qualifier, and one exclusion.

In [ ]:
CHICAGO_QUESTION = 'Which hotels in Chicago offer both a spa and a swimming pool?'
CHICAGO_PATTERN_NAME = 'Reviewed fixed Cypher: same-hotel spa AND pool filter'
CHICAGO_CONFIGURATION = 'city predicate, reviewed two-amenity AND filter'
CHICAGO_REQUESTED_FIELDS = ('hotel_name', 'guest_rating', 'amenities', 'source_filename')
CHICAGO_RECORD_FIELDS = (
    *CHICAGO_REQUESTED_FIELDS,
    'qualifies',
    'missing_required_amenities',
    'missing_requested_fields',
    'field_provenance',
    'structured_context_chars',
    'source_text_chars',
)
CHICAGO_PROVENANCE = {
    'source_chunk': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'source_filename': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'hotel_name': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'guest_rating': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'amenities': '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)',
}


def fixed_cypher_context(record):
    record['field_provenance'] = CHICAGO_PROVENANCE
    record['missing_requested_fields'] = [
        field for field in CHICAGO_REQUESTED_FIELDS
        if record.get(field) is None or record.get(field) == []
    ]
    structured_fields = {
        field: record.get(field)
        for field in (*CHICAGO_REQUESTED_FIELDS, 'qualifies', 'missing_required_amenities')
    }
    structured_chars, source_text_chars = context_char_counts(
        structured_fields, record.get('source_chunk') or ''
    )
    record['structured_context_chars'] = structured_chars
    record['source_text_chars'] = source_text_chars
    return record


candidate_records = [
    fixed_cypher_context(record) for record in chicago_filter_records(driver)
]

print(f'Question: {CHICAGO_QUESTION}')
print(f'Pattern: {CHICAGO_PATTERN_NAME}')
print(f'Configuration: {CHICAGO_CONFIGURATION}')
print(f'Reviewed Cypher: {CHICAGO_FILTER_QUERY}')
print(f'Parameters: city={CHICAGO_CITY!r}')
print(f'Expected sources: {list(CHICAGO_SOURCE_FILES)}')
print(f'Expected qualifier: {CHICAGO_QUALIFIER}')
print(f'Expected exclusion: {CHICAGO_EXCLUSION}, which has no spa and no swimming pool')
print(f'Candidate count: {len(candidate_records)}')
for record in candidate_records:
    verdict = 'qualifies' if record['qualifies'] else 'excluded'
    print(f'Candidate: {record["hotel_name"]} ({verdict})')
    print_record(record, CHICAGO_RECORD_FIELDS)
report_problems(
    chicago_filter_problems(candidate_records),
    'Two candidates, one qualifier, and the Windward exclusion are explicit.',
)

## 5. Compare passage search with Text2Cypher

Run the next cell to ask both read paths the same question: How many hotels are in Paris, and what is their average guest rating?

- **Passage path:** `search_hotel_knowledge` returns up to `HYBRID_TOP_K` passages ranked by relevance. This fixed limit controls the row count for every question.
- **Structured path:** `graph_query` creates Cypher from the question and fixed schema. It uses `EXPLAIN` to plan the query and allows only read-only statements. The query counts every matching `Hotel` record.
- **Comparison:** The output shows the ranked passage count beside the structured hotel count. Some ranked passages can describe hotels outside Paris.
- **Result:** Use structured queries for counts and averages. A fixed-size passage list represents only the top-ranked passages.

**Question design:** `GRAPH_QUERY_EXAMPLES` includes an example that calculates the average rating for Paris. This question also asks for a count. The added field requires a new aggregate query.

**Production note:** This workshop matches `paris` inside the free-text address without case sensitivity. A production graph should store and index a normalized city property. A read-only Neo4j user should also enforce read-only access.


In [ ]:
PARIS_QUESTION = (
    'How many hotels are in Paris, and what is their average guest rating?'
)
PARIS_TERM = 'paris'


def run_graph_query(question):
    """Run the shared structured path and report an expected failure.

    Module 3's `query_hotel_records` tool calls `graph_query`. That function
    plans generated Cypher with `EXPLAIN`, allows read-only statements, and
    limits list results to `MAX_GRAPH_QUERY_RECORDS` rows. This wrapper lets
    the notebook continue after a known service or model access error.
    """
    try:
        result = graph_query(question)
    except EXPECTED_QUERY_ERRORS as error:
        return {
            'cypher': '',
            'records': [],
            'error': f'{type(error).__name__}: {error}',
        }
    return {'cypher': result['cypher'], 'records': result['records'], 'error': None}


def paris_rows(passages):
    """Keep the ranked passages whose hotel address names Paris."""
    return [
        passage for passage in passages
        if PARIS_TERM in (passage['address'] or '').casefold()
    ]


def first_count(records):
    """Return the first whole-number column a generated aggregate returned.

    The model chooses the output column names. Find the count by its type.
    Neo4j returns count() as an integer and avg() as a float.
    """
    for record in records:
        for value in record.values():
            if isinstance(value, int) and not isinstance(value, bool):
                return value
    return None


ranked_passages = search_hotel_knowledge(PARIS_QUESTION)
ranked_paris_rows = paris_rows(ranked_passages)
graph_outcome = run_graph_query(PARIS_QUESTION)
counted_paris_hotels = first_count(graph_outcome['records'])

print(f'Question: {PARIS_QUESTION}')
print()
print('Passage path: search_hotel_knowledge, behind the search_hotel_passages tool')
print(f'  Rows returned: {len(ranked_passages)}, fixed by HYBRID_TOP_K={HYBRID_TOP_K}')
for rank, passage in enumerate(ranked_passages, 1):
    print(
        f'  Row {rank}: hotel={passage["hotel_name"]!r}, '
        f'address={passage["address"]!r}, '
        f'rating={passage["guest_rating"]!r}, '
        f'score={passage["combined_score"]:.6f}'
    )
print(
    f'  Ratings visible in those rows: '
    f'{[row["guest_rating"] for row in ranked_passages]}'
)
print()
print('Structured path: graph_query, behind the query_hotel_records tool')
print(f'  Generated Cypher: {graph_outcome["cypher"] or "none generated"}')
print(f'  Records: {graph_outcome["records"]}')
print(
    f'  List results are capped by the shipped path at '
    f'{MAX_GRAPH_QUERY_RECORDS} rows'
)
print(f'  Error: {graph_outcome["error"]}')
print()
print('Denominators side by side:')
print(f'  Rows the passage path returned:            {len(ranked_passages)}')
print(f'  Of those, rows whose address names Paris:  {len(ranked_paris_rows)}')
print(f'  Paris hotels the structured path counted:  {counted_paris_hotels}')
print(
    'Verdict: the ranking size is set by HYBRID_TOP_K and never by the number '
    'of hotels that match. It returns the same row count for a city with two '
    'hotels and for a city with two hundred, so the ratings it shows are not '
    'the denominator of the Paris average. Read the generated Cypher to '
    'confirm that the aggregate ran over every matching Hotel row.'
)


## Continue to Module 3

Module 3 gives the agent two read paths. Use the question type to choose between them.

| Question type | Read path | Evidence to inspect | Module 3 tool |
|---|---|---|---|
| Uses different wording | `VectorRetriever` | Ranked `Chunk` nodes, scores, and source paths | Basis for passage search |
| Includes an exact term | `HybridRetriever` | Exact-term hits and ranked `Chunk` nodes | Basis for passage search |
| Needs related hotel facts | `VectorCypherRetriever` | Source text, graph fields, relationships, and source paths | Basis for passage search |
| Has known fixed conditions | Reviewed fixed Cypher | Candidate, qualifier, and exclusion records | Reviewed workshop check |
| Needs source wording or facts about a few hotels | `HybridCypherRetriever` through `search_hotel_knowledge` | Up to five ranked passages and linked facts | `search_hotel_passages` |
| Needs counts, averages, rankings, filters, or relationship logic | Text2Cypher | Generated query, query plan, records, and errors | `query_hotel_records` |

- **Passage read path:** `search_hotel_passages` returns relevant source text and linked facts for one or a few hotels.
- **Structured read path:** `query_hotel_records` uses Text2Cypher across stored hotel records. Inspect the generated Cypher to confirm that it expresses the question correctly.
- **Module 3:** Registers both tools. The model reads their descriptions and selects a tool for each question.

In [ ]:
module_3_read_paths = (search_hotel_passages, query_hotel_records)
print('Module 3 read paths:')
for read_path in module_3_read_paths:
    print(f'  {read_path.tool_spec["name"]}')
driver.close()
print('Connection closed.')

# Section 5 can cache a second driver in workshop.hybrid_retrieval. Close any
# cached driver, then clear its cache. An empty cache needs no cleanup.
for cached_retriever in (
    hybrid_retrieval._get_retriever,
    hybrid_retrieval._get_graph_query_retriever,
):
    if cached_retriever.cache_info().currsize:
        cached_retriever().driver.close()
        cached_retriever.cache_clear()
hybrid_retrieval._get_driver.cache_clear()
print('Cached retrieval driver released.')
